# So sánh `cnn_only` và `cnn_lstm` trên `additional_obfu_merged.csv`

Notebook dùng hai checkpoint baseline chưa tuning:

- `cnn_only`: `cnn_only/artifacts_cnn_only_by_dataset/by_dataset/obfu_http/best_cnn_only.keras`
- `cnn_lstm`: `cnn_lstm/artifacts_cnn_lstm_by_dataset/by_dataset/obfu_http/best_hybrid_cnn_lstm.keras`

Cả hai dùng `MAX_LEN=768`, threshold cố định `0.5` và cùng dữ liệu test. Cell cuối chỉ tạo bảng so sánh hiệu suất tổng thể trên toàn bộ dataset đã merge.

Notebook được cấu hình `RUN_INFERENCE=True` để chạy lại toàn bộ inference bằng chính hai checkpoint trên; bước này gọi Python trong `.venv-webapp` và có thể mất vài phút trên CPU.


In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "analysis").is_dir() and (candidate / "cnn_only").is_dir() and (candidate / "cnn_lstm").is_dir():
            return candidate
    raise FileNotFoundError("Không tìm thấy thư mục gốc của project.")


PROJECT_ROOT = find_project_root(Path.cwd())
RESULT_DIR = PROJECT_ROOT / "reports" / "untuned_additional_obfu_evaluation"
DATASET_PATH = PROJECT_ROOT / "analysis" / "obfu_eval_outputs" / "additional_obfu_merged.csv"
EVALUATOR_PATH = PROJECT_ROOT / "analysis" / "evaluate_untuned_additional_obfu.py"
VENV_PYTHON = PROJECT_ROOT / ".venv-webapp" / "Scripts" / "python.exe"
AUDIT_PATH = RESULT_DIR / "evaluation_summary.json"
PREDICTIONS_PATH = RESULT_DIR / "predictions.csv"
COMPARISON_PATH = RESULT_DIR / "notebook_comparison_table.csv"

RUN_INFERENCE = True
THRESHOLD = 0.5
MAX_LEN = 768

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("RUN_INFERENCE:", RUN_INFERENCE)

PROJECT_ROOT: C:\Users\admin\Desktop\obfuscated-web-attack-detection
Dataset: C:\Users\admin\Desktop\obfuscated-web-attack-detection\analysis\obfu_eval_outputs\additional_obfu_merged.csv
RUN_INFERENCE: True


## 1. Chạy lại hoặc sử dụng kết quả inference đã kiểm chứng

Khi `RUN_INFERENCE=False`, notebook chỉ dùng cache nếu dataset, model và số dòng đều khớp với metadata kiểm tra. Nếu thiếu cache, evaluator sẽ tự chạy.

In [2]:
required_outputs = [DATASET_PATH, AUDIT_PATH, PREDICTIONS_PATH]
must_run = RUN_INFERENCE or not all(path.is_file() for path in required_outputs)

if must_run:
    if not VENV_PYTHON.is_file():
        raise FileNotFoundError(f"Không tìm thấy Python có TensorFlow: {VENV_PYTHON}")
    subprocess.run(
        [str(VENV_PYTHON), str(EVALUATOR_PATH)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Đã chạy lại inference cho cả hai model.")
else:
    print("Sử dụng dự đoán đã lưu của hai mô hình.")

Đã chạy lại inference cho cả hai model.


## 2. Xác minh dataset và checkpoint

In [3]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
provenance = audit["dataset_provenance"]

assert sha256(DATASET_PATH) == provenance["raw_merged_sha256"]
assert provenance["raw_merged_rows"] == 156186
assert provenance["training_overlap_rows"] == 0

model_rows = []
for model_name, contract in audit["models"].items():
    model_path = PROJECT_ROOT / contract["model_path"]
    assert model_path.is_file()
    assert sha256(model_path) == contract["model_sha256"]
    assert contract["max_len"] == MAX_LEN
    assert contract["threshold"] == THRESHOLD
    assert contract["threshold_strategy"] == "fixed"
    assert contract["tuning_directory"] is False
    assert "tuning" not in str(model_path).lower()
    model_rows.append(
        {
            "Mô hình": model_name,
            "Checkpoint": str(model_path),
            "Số tham số": contract["parameter_count"],
            "MAX_LEN": contract["max_len"],
            "Threshold": contract["threshold"],
        }
    )

model_table = pd.DataFrame(model_rows)
display(model_table.style.format({"Số tham số": "{:,.0f}", "Threshold": "{:.1f}"}).hide(axis="index"))

Mô hình,Checkpoint,Số tham số,MAX_LEN,Threshold
cnn_only,C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_only\artifacts_cnn_only_by_dataset\by_dataset\obfu_http\best_cnn_only.keras,"122,241",768,0.5
cnn_lstm,C:\Users\admin\Desktop\obfuscated-web-attack-detection\cnn_lstm\artifacts_cnn_lstm_by_dataset\by_dataset\obfu_http\best_hybrid_cnn_lstm.keras,"253,825",768,0.5


## 3. Thống kê `additional_obfu_merged.csv`

In [4]:
source_rows = []
for source_name, stats in provenance["sources"].items():
    source_rows.append(
        {
            "Nguồn": source_name,
            "Dòng đầu vào": stats["input_rows"],
            "Dòng rỗng loại bỏ": stats["empty_rows_removed"],
            "Dòng dùng để test": stats["output_rows"],
            "Normal": stats["label_counts"].get("0", 0),
            "Attack": stats["label_counts"].get("1", 0),
        }
    )

dataset_table = pd.DataFrame(source_rows)
dataset_table.loc[len(dataset_table)] = {
    "Nguồn": "Tổng",
    "Dòng đầu vào": dataset_table["Dòng đầu vào"].sum(),
    "Dòng rỗng loại bỏ": dataset_table["Dòng rỗng loại bỏ"].sum(),
    "Dòng dùng để test": dataset_table["Dòng dùng để test"].sum(),
    "Normal": dataset_table["Normal"].sum(),
    "Attack": dataset_table["Attack"].sum(),
}

display(dataset_table.style.format({
    "Dòng đầu vào": "{:,.0f}",
    "Dòng rỗng loại bỏ": "{:,.0f}",
    "Dòng dùng để test": "{:,.0f}",
    "Normal": "{:,.0f}",
    "Attack": "{:,.0f}",
}).hide(axis="index"))

print("Đầu vào mô hình lặp lại sau chuẩn hóa:", f"{provenance['repeated_model_inputs_retained']:,}")
print("Đầu vào xung đột nhãn:", provenance["model_inputs_with_conflicting_labels"])
print("Trùng với train OBFU_HTTP:", provenance["training_overlap_rows"])

Nguồn,Dòng đầu vào,Dòng rỗng loại bỏ,Dòng dùng để test,Normal,Attack
external_sql_obfuscated,"134,778",1,"134,777","70,575","64,202"
external_xss_obfuscated,"21,410",1,"21,409","3,531","17,878"
Tổng,"156,188",2,"156,186","74,106","82,080"


Đầu vào mô hình lặp lại sau chuẩn hóa: 10,687
Đầu vào xung đột nhãn: 0
Trùng với train OBFU_HTTP: 0


## 4. Tính lại metric từ xác suất dự đoán

In [5]:
predictions = pd.read_csv(PREDICTIONS_PATH)
required_columns = {
    "label", "source_dataset",
    "cnn_only_probability", "cnn_only_prediction",
    "cnn_lstm_probability", "cnn_lstm_prediction",
}
assert required_columns.issubset(predictions.columns)
assert len(predictions) == provenance["raw_merged_rows"]
assert predictions[list(required_columns)].isna().sum().sum() == 0
assert int(predictions["in_training_data"].sum()) == 0


def calculate_metrics(frame: pd.DataFrame, model_name: str, prefix: str, scope: str) -> dict:
    y_true = frame["label"].to_numpy(dtype=np.int8)
    y_prob = frame[f"{prefix}_probability"].to_numpy(dtype=np.float64)
    y_pred = (y_prob >= THRESHOLD).astype(np.int8)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "Phạm vi": scope,
        "Mô hình": model_name,
        "Số mẫu": len(frame),
        "MAX_LEN": MAX_LEN,
        "Threshold": THRESHOLD,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision Attack": precision,
        "Recall Attack": recall,
        "F1 Attack": f1,
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "FP": int(fp),
        "FN": int(fn),
        "Tổng lỗi": int(fp + fn),
    }


scopes = [("Tổng thể", predictions)]
source_labels = {
    "external_sql_obfuscated": "SQLi obfuscated",
    "external_xss_obfuscated": "XSS obfuscated",
}
for source, source_frame in predictions.groupby("source_dataset", sort=False):
    scopes.append((source_labels.get(source, source), source_frame))

rows = []
for scope_name, frame in scopes:
    rows.append(calculate_metrics(frame, "cnn_only", "cnn_only", scope_name))
    rows.append(calculate_metrics(frame, "cnn_lstm", "cnn_lstm", scope_name))

comparison_table = pd.DataFrame(rows)
comparison_table.to_csv(COMPARISON_PATH, index=False, encoding="utf-8")
print("Đã kiểm tra", f"{len(predictions):,}", "dự đoán và lưu bảng tại:", COMPARISON_PATH)

Đã kiểm tra 156,186 dự đoán và lưu bảng tại: C:\Users\admin\Desktop\obfuscated-web-attack-detection\reports\untuned_additional_obfu_evaluation\notebook_comparison_table.csv


## 5. Bảng so sánh hiệu suất

In [6]:
# CELL CUỐI: bảng hiệu suất của cnn_only và cnn_lstm trên cùng dữ liệu và threshold 0,5.
final_columns = [
    "Phạm vi", "Mô hình", "Số mẫu", "MAX_LEN", "Threshold",
    "Accuracy", "Precision Attack", "Recall Attack", "F1 Attack",
    "ROC-AUC", "PR-AUC", "FP", "FN", "Tổng lỗi",
]
final_table = comparison_table.loc[comparison_table["Phạm vi"].eq("Tổng thể"), final_columns].copy()

display(
    final_table.style
    .format(
        {
            "Số mẫu": "{:,.0f}",
            "MAX_LEN": "{:,.0f}",
            "Threshold": "{:.1f}",
            "Accuracy": "{:.2%}",
            "Precision Attack": "{:.2%}",
            "Recall Attack": "{:.2%}",
            "F1 Attack": "{:.2%}",
            "ROC-AUC": "{:.2%}",
            "PR-AUC": "{:.2%}",
            "FP": "{:,.0f}",
            "FN": "{:,.0f}",
            "Tổng lỗi": "{:,.0f}",
        }
    )
    .hide(axis="index")
    .set_caption("So sánh cnn_only và cnn_lstm trên additional_obfu_merged.csv")
)


Phạm vi,Mô hình,Số mẫu,MAX_LEN,Threshold,Accuracy,Precision Attack,Recall Attack,F1 Attack,ROC-AUC,PR-AUC,FP,FN,Tổng lỗi
Tổng thể,cnn_only,"156,186",768,0.5,77.66%,84.85%,69.98%,76.70%,86.46%,87.84%,"10,253","24,644","34,897"
Tổng thể,cnn_lstm,"156,186",768,0.5,80.89%,87.74%,73.97%,80.27%,90.06%,89.54%,"8,484","21,367","29,851"
